# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [1]:
# Clear prior data. 
import os, sys, shutil

# Add parent directory to Python path to import EnvironmentData.
sys.path.append(os.path.dirname(os.getcwd()))

# Get the EnvironmentData class.
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('../data'):
    shutil.rmtree('../data')
    os.makedirs('../data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    #days_back = 365 * 2,
    days_back = 7,
    coris_enabled = True,
    licor_enabled = True,
    conserv_enabled = True, 
    testing = True,
    # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
    home_directory = ".."
)

DEBUG: Enabled data sources: ['Coris', 'Conserv', 'LI-COR']


Gathering LI-COR readings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.65it/s]


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('../data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-11-15 15:23:27,379 - EnvironmentData - INFO - Initialized Conserv client with 5 customers
2025-11-15 15:23:27,380 - EnvironmentData - INFO - Enabled data sources: ['Coris', 'Conserv', 'LI-COR']
2025-11-15 15:23:27,380 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/cats/user/?ApiKey=XXXX&CatsUserID=XXXX
2025-11-15 15:23:29,376 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21373&ReadingType=SensorReadingF&StartUTC=1762640607&EndUTC=1763245407&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-15 15:23:31,204 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21375&ReadingType=SensorReadingF&StartUTC=1762640607&EndUTC=1763245407&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-15 15:23:33,012 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21377&Readin

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1762640607,1763245409,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.589996,null,true
1762641207,1763245409,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.519997,null,true
1762641807,1763245409,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.580002,null,true
1762642407,1763245409,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.5,null,true
1762643007,1763245409,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.5,null,true


In [4]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "LI-COR").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1762641000,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",71.184517,null,true
1762641900,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",71.184517,null,true
1762642800,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",71.261734,null,true
1762643700,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",71.338959,null,true
1762644600,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",71.338959,null,true


In [5]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Conserv").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1762641139,1763245420,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.512001,null,true
1762642033,1763245420,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.800003,null,true
1762642933,1763245420,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",70.052002,null,true
1762643833,1763245420,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",70.484001,null,true
1762644733,1763245420,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",70.286003,null,true


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [6]:
envdt.get_current_readings()

# Data is read into new-readings folder for consolidation at the end of the day.
import os
filename = os.listdir('../data/new-readings')[0]
print(filename)
polars.read_parquet('../data/new-readings/' + filename).sample(5)

Gathering Conserv current readings: 100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:40<00:00, 100.57s/it]


1763245533.parquet


SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1763244728,1763245536,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,52.330002,false
1763244967,1763245536,"""Conserv""","""conserv:333:c008944""","""BYCBA_0200201N8_N___""","""conserv:333:c008944:RH""","""BYCBA_0200201N8_N___ - RH""","""RH""",null,47.779999,false
1763244955,1763245536,"""Conserv""","""conserv:333:c008914""","""BYCBA_030030104_S___""","""conserv:333:c008914:RH""","""BYCBA_030030104_S___ - RH""","""RH""",null,44.939999,false
1763245357,1763245536,"""Conserv""","""conserv:333:c009016""","""BYCBA_0200216_E_____""","""conserv:333:c009016:Temperatur…","""BYCBA_0200216_E_____ - Tempera…","""Temperature""",72.428001,null,false
1763245098,1763245536,"""Conserv""","""conserv:333:c009081""","""BCSC__01H103________""","""conserv:333:c009081:RH""","""BCSC__01H103________ - RH""","""RH""",null,49.02,false


# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [7]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
print(os.listdir('../data/new-readings'))

[]


**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [8]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [9]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('../data/sensor_readings.parquet')
sensor_readings.head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1762640824,1763245420,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,49.779999,null,true
1762641724,1763245420,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,50.439999,null,true
1762642624,1763245420,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,49.740002,null,true
1762643524,1763245420,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,49.310001,null,true
1762644424,1763245420,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,49.59,null,true


In [10]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1763239500,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",70.682579,null,null,true
1763240400,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",70.721191,null,null,true
1763241300,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",70.643967,null,null,true
1763242200,1763245533,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""",null,"""Temperature""",70.643967,null,null,true
1763245636,1763245636,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.643967,null,null,false


In [11]:
# Device Readings.
device_readings = polars.read_parquet('../data/device_readings.parquet')
device_readings.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
str,str,str,str,str,str,i64,i32,bool,f32,f32
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH, conser…","""BYCBA_0400410__N____ - RH, BYC…","""RH, Temperature""",1762640824,1763245420,true,70.592003,49.779999
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH, conser…","""BYCBA_0400410__N____ - RH, BYC…","""RH, Temperature""",1762641724,1763245420,true,70.484001,50.439999
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH, conser…","""BYCBA_0400410__N____ - RH, BYC…","""RH, Temperature""",1762642624,1763245420,true,70.375999,49.740002
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH, conser…","""BYCBA_0400410__N____ - RH, BYC…","""RH, Temperature""",1762643524,1763245420,true,70.501999,49.310001
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH, conser…","""BYCBA_0400410__N____ - RH, BYC…","""RH, Temperature""",1762644424,1763245420,true,70.447998,49.59


In [12]:
# Sensors
sensors = polars.read_parquet('../data/sensors.parquet')
sensors.head()

Source,SensorID,SensorName,SensorType,DeviceID,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection,DeviceName,SensorReadingUTC,QueryUTC,Historical
str,str,str,str,str,str,str,str,str,str,str,i64,i32,bool
"""LI-COR""","""licor:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""","""licor:10740550""","""RH""","""FLOATER""","""Unknown""","""FLOATER""",null,"""ICSC__010C149_______""",1763245636,1763245636,false
"""LI-COR""","""licor:10740550-10740550-1""","""ICSC__010C149________Temperatu…","""Temperature""","""licor:10740550""","""Temperature""","""FLOATER""","""Unknown""","""FLOATER""",null,"""ICSC__010C149_______""",1763245636,1763245636,false
"""Coris""","""coris:21378""","""RH KGL 21_D0B2""","""Humidity""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated""","""Peabody TH-L Upper Great Hall …",1763033607,1763245418,true
"""Coris""","""coris:21378""","""RH KGL 21_D0B2""","""Humidity""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated""","""Peabody TH-L Upper Great Hall …",1763048007,1763245418,true
"""Coris""","""coris:21378""","""RH KGL 21_D0B2""","""Humidity""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated""","""Peabody TH-L Upper Great Hall …",1762722807,1763245418,true


In [13]:
# Devices. 
devices = polars.read_parquet('../data/devices.parquet')
devices.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str,str
"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-2, lic…","""ICSC__010C149________RH, ICSC_…","""RH, Temperature""","""RH""","""FLOATER""","""Unknown""","""FLOATER""",null
"""Coris""","""coris:12167""","""Peabody TH-L Upper Great Hall …","""coris:21378, coris:21378, cori…","""RH KGL 21_D0B2, RH KGL 21_D0B2…","""Humidity, Humidity, Humidity, …","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""Conserv""","""conserv:333:c009072""","""BBARCH0100001_______""","""conserv:333:c009072:RH, conser…","""- RH, - RH, - RH, - RH, - RH, …","""RH, RH, RH, RH, RH, RH, RH, RH…",null,"""MALFORMED""","""Unknown""","""Unknown""",null
"""Conserv""","""conserv:333:c009073""","""BBARCHB100116_______""","""conserv:333:c009073:RH, conser…","""- RH, - RH, - RH, - RH, - RH, …","""RH, RH, RH, RH, RH, RH, RH, RH…",null,"""MALFORMED""","""Unknown""","""Unknown""",null
"""Conserv""","""conserv:333:c009081""","""BCSC__01H103________""","""conserv:333:c009081:RH, conser…","""- RH, - RH, - RH, - RH, - RH, …","""RH, RH, RH, RH, RH, RH, RH, RH…",null,"""MALFORMED""","""Unknown""","""Unknown""",null


In [14]:
# UTC Date/Time Info
utcs = polars.read_parquet('../data/utcs.parquet').head()
utcs.head()

UTC,datetime_utc,datetime_est,date,time,year,month,day_of_week,day_of_week_monday1_sunday7,hour_24,hour_12,am_pm
i64,datetime[μs],"datetime[μs, America/New_York]",date,time,i32,i8,str,i8,i8,i8,str
1762918403,2025-11-11 20:33:23,2025-11-11 15:33:23 EST,2025-11-11,15:33:23,2025,11,"""Tuesday""",2,15,3,"""PM"""
1762787332,2025-11-10 08:08:52,2025-11-10 03:08:52 EST,2025-11-10,03:08:52,2025,11,"""Monday""",1,3,3,"""AM"""
1762918407,2025-11-11 20:33:27,2025-11-11 15:33:27 EST,2025-11-11,15:33:27,2025,11,"""Tuesday""",2,15,3,"""PM"""
1763180558,2025-11-14 21:22:38,2025-11-14 16:22:38 EST,2025-11-14,16:22:38,2025,11,"""Friday""",5,16,4,"""PM"""
1762787350,2025-11-10 08:09:10,2025-11-10 03:09:10 EST,2025-11-10,03:09:10,2025,11,"""Monday""",1,3,3,"""AM"""


In [15]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('../data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

Source,date,SensorID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-11-15,"""conserv:333:c008733:RH""",1,0.0,44.77,null,44.77,null,44.77
"""Conserv""",2025-11-15,"""conserv:333:c008733:Temperatur…",1,72.248001,0.0,72.248001,null,72.248001,null
"""Conserv""",2025-11-15,"""conserv:333:c008781:RH""",1,0.0,47.709999,null,47.709999,null,47.709999
"""Conserv""",2025-11-15,"""conserv:333:c008781:Temperatur…",1,71.690002,0.0,71.690002,null,71.690002,null
"""Conserv""",2025-11-15,"""conserv:333:c008784:RH""",1,0.0,45.080002,null,45.080002,null,45.080002


In [16]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('../data/device_readings_daily.parquet')
device_readings_daily.head()

Source,date,DeviceID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-11-15,"""conserv:333:c008733""",1,72.248001,44.77,72.248001,44.77,72.248001,44.77
"""Conserv""",2025-11-15,"""conserv:333:c008781""",1,71.690002,47.709999,71.690002,47.709999,71.690002,47.709999
"""Conserv""",2025-11-15,"""conserv:333:c008784""",1,71.959999,45.080002,71.959999,45.080002,71.959999,45.080002
"""Conserv""",2025-11-15,"""conserv:333:c008924""",1,71.816002,45.310001,71.816002,45.310001,71.816002,45.310001
"""Conserv""",2025-11-15,"""conserv:333:c008949""",1,71.545998,44.669998,71.545998,44.669998,71.545998,44.669998


In [17]:
# Differentiate historical vs. cron readings by filtering on Historical = true.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('../data/device_readings.parquet') 
    WHERE Historical
    LIMIT 5
""").to_df()

,Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
0,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:RH, conserv:333:c008706:Te...","BYCBA_0400410__N____ - RH, BYCBA_0400410__N___...","RH, Temperature",1762640824,1763245420,True,70.592003,49.779999
1,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:RH, conserv:333:c008706:Te...","BYCBA_0400410__N____ - RH, BYCBA_0400410__N___...","RH, Temperature",1762641724,1763245420,True,70.484001,50.439999
2,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:RH, conserv:333:c008706:Te...","BYCBA_0400410__N____ - RH, BYCBA_0400410__N___...","RH, Temperature",1762642624,1763245420,True,70.375999,49.740002
3,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:RH, conserv:333:c008706:Te...","BYCBA_0400410__N____ - RH, BYCBA_0400410__N___...","RH, Temperature",1762643524,1763245420,True,70.501999,49.310001
4,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:RH, conserv:333:c008706:Te...","BYCBA_0400410__N____ - RH, BYCBA_0400410__N___...","RH, Temperature",1762644424,1763245420,True,70.447998,49.590000


Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.